# PONITA + ECT on Colab (resumable)

Trains all four PONITA variants (**baseline, concat, film-final, leap**) on MD17 aspirin, 2000 epochs each, with target normalization on.

## Why this won't lose your progress
- Checkpoints (model + optimizer + scheduler + epoch) are written to **Google Drive** every 25 epochs, atomically.
- The training script **auto-resumes** from the last checkpoint. If Colab disconnects, re-run cells 2, 3 and 5 and it continues where it stopped.
- The full log is also appended to a file in the checkpoint folder on Drive.
- A checkpoint saved with a different `--epochs` value is ignored and that model starts fresh, because the learning-rate schedule depends on it.

## To avoid disconnects in the first place
- **Colab Pro+** allows background execution and gives faster GPUs.
- On free tier: keep the tab open and the laptop awake. Expect several resume cycles.
- Check which GPU you got in cell 1. An A100 or L4 is much faster than a T4.

### One-time setup (do this before running anything)
Upload your project folder (the `dino` repo — everything: `train_ponita_md17.py`, `models/`, `EGNN.py`, `ECT.py`, `qm9_ect.py`, `mesh_lib/`, …) to Google Drive, e.g. to `MyDrive/dino`. Then set `REPO_DIR` below to match.

In [ ]:
# Cell 1 — confirm you have a GPU
!nvidia-smi

In [ ]:
# Cell 2 — mount Drive and set paths (re-run this after any reconnect)
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/dino'          # where you uploaded the repo
CKPT_DIR = '/content/drive/MyDrive/dino_ckpts'    # checkpoints persist here
DATA_DIR = '/content/drive/MyDrive/dino_data/md17' # MD17 downloads here once
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
assert os.path.exists(os.path.join(REPO_DIR, 'train_ponita_md17.py')), \
    f'train_ponita_md17.py not found in {REPO_DIR} — fix REPO_DIR'
print('Repo :', REPO_DIR)
print('Ckpts:', CKPT_DIR)
print('Data :', DATA_DIR)

In [ ]:
# Cell 3 — install dependencies matching Colab's torch build
# (ponita needs Python 3.10+, which Colab has — so NONE of the cluster's
#  match/case patches are needed here.)
import torch
v = torch.__version__.split('+')[0]
cu = ('cu' + torch.version.cuda.replace('.', '')) if torch.version.cuda else 'cpu'
wheel = f'https://data.pyg.org/whl/torch-{v}+{cu}.html'
print('torch', torch.__version__, '->', wheel)
!pip -q install pyg-lib -f {wheel}
!pip -q install torch-scatter -f {wheel}
!pip -q install torch-geometric
!pip -q install git+https://github.com/ebekkers/ponita.git

In [ ]:
# Cell 4 — sanity check: imports work and equivariance holds
import os
os.chdir(REPO_DIR)
!python test_equivariance.py

In [ ]:
# Cell 5 — leap with no local cutoff (resumes automatically). Re-run after any reconnect.
# Tests whether leap's hard 5 Å neighbor cutoff is what hurts its forces.
# Same settings as the seed-0 run of the other models; only --local-cutoff differs.
import os
os.chdir(REPO_DIR)
EPOCHS = 2000
SEED   = 0
LOG    = f'{CKPT_DIR}/ponita_leap_rc1000_seed{SEED}_ep{EPOCHS}.log'   # log survives disconnects
!python -u train_ponita_md17.py \
    --molecule aspirin --model ect --cond-type leap --local-cutoff 1000 \
    --epochs {EPOCHS} --seed {SEED} \
    --data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" \
    --log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 \
    --lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 \
    --basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 \
    --icosphere-nu 3 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 6 — concat-local: mean-pooled local ECT, concat injection (resumes automatically)
# Local ECT per atom → mean pool → graph token → broadcast & concat at input.
# Tests whether local neighborhood descriptors aggregated globally beat the whole-molecule ECT.
import os
os.chdir(REPO_DIR)
EPOCHS = 2000
SEED   = 0
LOG    = f'{CKPT_DIR}/ponita_concat_local_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py \
    --molecule aspirin --model ect --cond-type concat-local \
    --epochs {EPOCHS} --seed {SEED} \
    --data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" \
    --log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 \
    --lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 \
    --basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 \
    --icosphere-nu 3 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 7 — film-local: mean-pooled local ECT, FiLM injection (resumes automatically)
# Local ECT per atom → mean pool → graph token → bounded FiLM at graph level.
import os
os.chdir(REPO_DIR)
EPOCHS = 2000
SEED   = 0
LOG    = f'{CKPT_DIR}/ponita_film_local_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py \
    --molecule aspirin --model ect --cond-type film-local \
    --epochs {EPOCHS} --seed {SEED} \
    --data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" \
    --log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 \
    --lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 \
    --basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 \
    --icosphere-nu 3 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 8 — concat-mlp: global ECT + flat MLP encoder (resumes automatically)
# Same as concat but with the local variants' MLP encoder. Run this one first.
import os
os.chdir(REPO_DIR)
EPOCHS = 2000
SEED   = 0
LOG    = f'{CKPT_DIR}/ponita_concat_mlp_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py \
    --molecule aspirin --model ect --cond-type concat-mlp \
    --epochs {EPOCHS} --seed {SEED} \
    --data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" \
    --log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 \
    --lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 \
    --basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 \
    --icosphere-nu 3 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 9 — concat-local-branch: local ECT + ECTBranch encoder (resumes automatically)
# Same as concat-local but with the global variants' ECTBranch encoder.
import os
os.chdir(REPO_DIR)
EPOCHS = 2000
SEED   = 0
LOG    = f'{CKPT_DIR}/ponita_concat_local_branch_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py \
    --molecule aspirin --model ect --cond-type concat-local-branch \
    --epochs {EPOCHS} --seed {SEED} \
    --data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" \
    --log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 \
    --lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 \
    --basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 \
    --icosphere-nu 3 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 10 — concat-adaptive: input concat + AdaLN and adaptive scale in every block (resumes automatically)
# Global ECT + ECTBranch. concat plus Proteina-style conditioning in all 5 PONITA blocks. Run first.
import os
os.chdir(REPO_DIR)
EPOCHS = 2000
SEED   = 0
LOG    = f'{CKPT_DIR}/ponita_concat_adaptive_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py \
    --molecule aspirin --model ect --cond-type concat-adaptive \
    --epochs {EPOCHS} --seed {SEED} \
    --data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" \
    --log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 \
    --lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 \
    --basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 \
    --icosphere-nu 3 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 11 — adaptive: AdaLN and adaptive scale in every block, no input concat (resumes automatically)
# Same conditioning without the input concat, to see whether the per-block conditioning works on its own.
import os
os.chdir(REPO_DIR)
EPOCHS = 2000
SEED   = 0
LOG    = f'{CKPT_DIR}/ponita_adaptive_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py \
    --molecule aspirin --model ect --cond-type adaptive \
    --epochs {EPOCHS} --seed {SEED} \
    --data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" \
    --log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 \
    --lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 \
    --basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 \
    --icosphere-nu 3 2>&1 | tee -a "{LOG}"

In [ ]:
# Cell 12 — concat-adaptive-glu: Cell 10 plus an ECT gate on each MLP hidden layer (resumes automatically)
# Same as concat-adaptive, plus an adaptive GLU gate. Compare with Cell 10.
import os
os.chdir(REPO_DIR)
EPOCHS = 2000
SEED   = 0
LOG    = f'{CKPT_DIR}/ponita_concat_adaptive_glu_seed{SEED}_ep{EPOCHS}.log'
!python -u train_ponita_md17.py \
    --molecule aspirin --model ect --cond-type concat-adaptive-glu \
    --epochs {EPOCHS} --seed {SEED} \
    --data-root "{DATA_DIR}" --ckpt-dir "{CKPT_DIR}" \
    --log-interval 25 --warmup 100 --batch-size 5 --lr 5e-4 \
    --lambda-f 500 --hidden-dim 128 --num-layers 5 --num-ori 20 \
    --basis-dim 256 --ponita-radius 1000.0 --ect-out-dim 64 \
    --icosphere-nu 3 2>&1 | tee -a "{LOG}"

### If it disconnects
1. Reconnect the runtime.
2. Re-run **Cell 2** (remount Drive), **Cell 3** (reinstall), then whichever cell was running.
3. You'll see `Resumed from ..._last.pt at epoch N` and it continues.

### Cells and what they test
| Cell | Model | What it tests |
|---|---|---|
| 5 | leap rc1000 | Does removing the hard 5 Å cutoff fix leap's forces? (done: no) |
| 6 | concat-local | Local ECT pooled to a graph token, concat injection |
| 7 | film-local | Local ECT pooled to a graph token, FiLM injection |
| 8 | concat-mlp | Encoder swap: global ECT + flat MLP |
| 9 | concat-local-branch | Encoder swap: local ECT + ECTBranch |
| 10 | concat-adaptive | concat + AdaLN / adaptive scale in every PONITA block |
| 11 | adaptive | AdaLN / adaptive scale in every block, no input concat |
| 12 | concat-adaptive-glu | Cell 10 plus an ECT gate on each MLP hidden layer |

Run cells one at a time (not Run all).

### Extra seeds
Set `SEED = 1` then `SEED = 2` and re-run the cells for the models that perform best.